# 16 — Preferences, Reward Models, and RLHF

**Network LLM Engineering — Part IV — Post-Training**

### Learning goals
- Understand preference data and reward modeling
- Understand the classic RLHF pipeline
- Know when preferences add value beyond SFT

In [ ]:
from pathlib import Path

def find_root():
    for p in [Path.cwd(), Path.cwd().parent, Path("/content/network_llm_engineering_course")]:
        if (p / "data" / "glossary.csv").exists():
            return p
    raise FileNotFoundError("Run from the extracted network_llm_engineering_course folder.")

ROOT = find_root()
DATA = ROOT / "data"
print("Course root:", ROOT)

## Why preferences?

SFT says: **this answer is desired**.
Preference data says: **for the same prompt, answer A is better than answer B**.

This is useful when the desired property is comparative:
- cautious vs reckless troubleshooting,
- concise vs bloated,
- evidence-based vs speculative,
- policy-compliant vs policy-violating.

In [ ]:
import json
prefs = [json.loads(x) for x in open(DATA/"network_preferences.jsonl", encoding="utf-8")]
p = prefs[0]
print("PROMPT:", p["prompt"][0]["content"])
print("\nCHOSEN:", p["chosen"][0]["content"])
print("\nREJECTED:", p["rejected"][0]["content"])
print("\nPRINCIPLE:", p["principle"])

## Classic RLHF pipeline

`pretrained/instruct model -> SFT -> human preference pairs -> reward model -> RL (often PPO) -> aligned policy`

A **reward model** learns a scalar preference signal.
**PPO** then updates the policy to increase reward while typically constraining drift from a reference policy.

This pipeline is powerful but operationally complex. Direct preference methods such as DPO were created partly to simplify it.

## Reward hacking

If the reward is incomplete, the model may optimize the metric instead of the real intent.
Example: reward +1 whenever the answer contains "verify". The model learns to spam the word "verify" without improving diagnosis.

For networking, prefer **verifiable state checks** whenever possible.

### Exercise

Write one bad reward function for network troubleshooting and explain how the model could exploit it.
Then propose a better composite/verifiable reward.